## pytorch归一化

Pytorch中的**归一化方式**主要分为以下几种：

* BatchNorm（2015年）
* LayerNorm（2016年）
* InstanceNorm（2017年）
* GroupNorm（2018年）
* RMSNorm (2023)

### BatchNorm2D
- **归一化维度**: 对单个通道，跨 mini-batch 的所有样本计算均值和方差。
- **公式**：

\begin{align}\mu_B=\frac{1}{m H W} \sum_{n=1}^m \sum_{i=1}^H \sum_{j=1}^W x_{n, c, i, j} \\ \sigma_B^2=\frac{1}{m H W} \sum_{n=1}^m \sum_{i=1}^H \sum_{j=1}^W\left(x_{n, c, i, j}-\mu_B\right)^2 \\ \hat{x}_{n, c, i, j}=\frac{x_{n, c, i, j}-\mu_B}{\sqrt{\sigma_B^2+\epsilon}} \\ y_{n, c, i, j}=\gamma_c \hat{x}_{n, c, i, j}+\beta_c\end{align}

其中$\mu_B$是均值计算，$\sigma_B^2$是方差计算，$hat{x}_{n, c, i, j}$是归一化计算，$y_{n, c, i, j}$是仿射变化。 归一化计算分母中的 ϵ 是一个非常小的数，作用是防止数值计算不稳定。 γ 和 β 是仿射参数，将归一化后的数据再次放缩得到新的数据， γ 可以理解为标准差， β 可以理解为均值，它们两个一般是可学习的。可以发现， γ 和 β 是BatchNorm2D层仅有的可学习参数。

说明：

它是 **沿着输入的第二维** （即channel维度）算均值和方差的。比如输入大小为 (N,C,H,W) ，**则均值 E[x] 为 input.mean((0,2,3))** 。

![](https://ones.ainewera.com/wiki/api/wiki/editor/JNwe8qUX/RDYCnwvo/resources/sJfM3hG5bF8a_AX4LEvVpR1C8wZ5bNstHzCnojogIbI.png?token=W.h2ZbJ4tknyp15AZ3wAqFjz9AVke9xNe8MgBZ16uEBjRjHAXghPTeHOCSS3jem-A)

代码：

```
torch.nn.BatchNorm2d(num_features, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True, device=None, dtype=None)
```

主要参数的含义：

* num_features：通道数C。
* eps：归一化时加到分母上，防止分母过小导致数值计算不稳定。**不用管这一项，一般用默认的就好。**
* momentum：在track_running_stats为True时，momentum是训练过程中对均值和方差进行动量更新的动量参数；在track_running_stats为False时，momentum不起作用。
* affine：当为True时， γ 和 β 参数是可学习的；反之，是不可学习的。
* track_running_stats：当为True时，在训练时会始终记录并更新（通过动量方法更新）全局的均值和方差，然后在**测试**时可以用这个均值和方差来归一化（**为什么要这样做？你可以理解为这个均值和方差是所有****训练样本****的均值和方差，是全局的，对整个样本集的统计信息的描述更加准确一些** ）；当为False时，不记录更新全局的均值和方差，这样的话，**测试**时用那个batch的测试数据本身的样本和方差来归一化。

输入输出的维度：

输入维度：(N,C,H,W) (N为batchsize，下文不再赘述)

输出维度：(N,C,H,W)

BatchNorm1D

它也是 **沿着输入的第二维** （即channel维）算均值和方差的。它与BatchNorm2D的不同之处在于，它的输入和输出维度可以是更低：

输入维度：(N,C) 或 (N,C,L)

输出维度：(N,C) 或 (N,C,L)

因此，它又可以被称为是**时间BN**

BatchNorm3D

它也是 **沿着输入的第二维** （即channel维）算均值和方差的。它与BatchNorm2D的不同之处在于，它的输入和输出维度可以是更高：

输入维度： (N,C,D,H,W)

输出维度： (N,C,D,H,W)

因此，它又可以被称为是**体积BN**或者**时空BN**

In [10]:
import torch
import torch.nn as nn

# NLP Example
embedding = torch.tensor([[[1,2,3,4]]], dtype=torch.float32)
batch, sentence_length, embedding_dim = embedding.shape  # 1,1,4
mean = embedding.mean(dim=(0,2), keepdim=True)  # 2.5
std = embedding.std(dim=(0,2), keepdim=True)    # 1.2910
batch_norm = nn.BatchNorm1d(sentence_length)
# Activate module
norm_embedding = batch_norm(embedding)       # [[[-1.3416, -0.4472,  0.4472,  1.3416]]]
print("mean:", mean)
print("std:", std)
print("norm1d embdedding:", norm_embedding)

mean: tensor([[[2.5000]]])
std: tensor([[[1.2910]]])
norm1d embdedding: tensor([[[-1.3416, -0.4472,  0.4472,  1.3416]]],
       grad_fn=<NativeBatchNormBackward0>)


In [11]:
# Image Example
input = torch.tensor([[[[ 1,  2],
                        [3,  4]],
                       [[ 2,  2],
                        [3,  4]]]], dtype=torch.float32)
N, C, H, W = input.shape   # [N, C, H, W]: [1,2,2,2]
print(input.shape)
# Normalize over the C dimensions
mean = input.mean(dim=(0,2,3), keepdim=True)  # [2.5,2.75]
std = input.std(dim=(0,2,3), keepdim=True)    # [1.2910, 0.9574]
batch_norm = nn.BatchNorm2d(C)
output1 = batch_norm(input)  #[[[[-1.3416, -0.4472],[ 0.4472,  1.3416]]],
                             # [[[-0.9045, -0.9045],[ 0.3015,  1.5075]]]]
print("mean:", mean)
print("std:", std)
print("norm2d embedding:", output1)

torch.Size([1, 2, 2, 2])
mean: tensor([[[[2.5000]],

         [[2.7500]]]])
std: tensor([[[[1.2910]],

         [[0.9574]]]])
norm2d embedding: tensor([[[[-1.3416, -0.4472],
          [ 0.4472,  1.3416]],

         [[-0.9045, -0.9045],
          [ 0.3015,  1.5075]]]], grad_fn=<NativeBatchNormBackward0>)


In [12]:
# Image Example
input = torch.tensor([[[[ 1,  2],
                        [3,  4]],
                       ],
                      [[[ 1,  2],
                        [3,  4]],
                       ]], dtype=torch.float32)
N, C, H, W = input.shape   # [N, C, H, W]: [2,1,2,2]
print(input.shape)
# Normalize over the C dimensions
mean = input.mean(dim=(0,2,3), keepdim=True)  # [2.5,2.75]
std = input.std(dim=(0,2,3), keepdim=True)    # [1.2910, 0.9574]
batch_norm = nn.BatchNorm2d(C)
output1 = batch_norm(input)  #[[[[-1.3416, -0.4472],[ 0.4472,  1.3416]]],
                             # [[[-0.9045, -0.9045],[ 0.3015,  1.5075]]]]
print("mean:", mean)
print("std:", std)
print("norm2d embedding:", output1)

torch.Size([2, 1, 2, 2])
mean: tensor([[[[2.5000]]]])
std: tensor([[[[1.1952]]]])
norm2d embedding: tensor([[[[-1.3416, -0.4472],
          [ 0.4472,  1.3416]]],


        [[[-1.3416, -0.4472],
          [ 0.4472,  1.3416]]]], grad_fn=<NativeBatchNormBackward0>)


### LayerNorm
- **归一化维度**: 针对单个样本的所有通道 𝐶 和空间维度 𝐻×𝑊一起计算均值和方差。
- **公式**:

\begin{align}\mu_L=\frac{1}{C H W} \sum_{c=1}^C \sum_{i=1}^H \sum_{j=1}^W x_{n, c, i, j} \\ \sigma_L^2=\frac{1}{C H W} \sum_{c=1}^C \sum_{i=1}^H \sum_{j=1}^W\left(x_{n, c, i, j}-\mu_L\right)^2 \\ \hat{x}_{n, c, i, j}=\frac{x_{n, c, i, j}-\mu_L}{\sqrt{\sigma_L^2+\epsilon}} \\ y_{n, c, i, j}=\gamma_c \hat{x}_{n, c, i, j}+\beta_c\end{align}


![](https://ones.ainewera.com/wiki/api/wiki/editor/JNwe8qUX/RDYCnwvo/resources/s0Y8rb94cXyXsjlra75GjwCzQoNsq6FW2uhiyM9EU_c.png?token=W.h2ZbJ4tknyp15AZ3wAqFjz9AVke9xNe8MgBZ16uEBjRjHAXghPTeHOCSS3jem-A)

代码：

```
torch.nn.LayerNorm(normalized_shape, eps=1e-05, elementwise_affine=True, device=None, dtype=None)
```
在pytorch实现中，layerNorm更加灵活，可以**对输入的后几维（具体是几维取决于初始化参数normalized_shape）合并在一起**算均值和方差的。比如输入大小为 (N,C,H,W) （像图像一样），若normalized_shape为三维，则均值 E[x] 为 input.mean((−3,−2,−1)) 。比如输入大小为 (N,L,D) （像文本一样），若normalized_shape为一维，则均值 E[x] 为 input.mean((−1)) ，即Transformer中使用的LayerNorm。

主要参数的含义：

* normalized_shape：LayerNorm的输入的大小（除去第一维batchsize维度）。比如想让LayerNorm的输入大小为 (N,C,H,W) ，那么normalized_shape可以是一个 [C,H,W] 的list。
* eps：归一化时加到分母上，防止分母过小导致数值计算不稳定。**不用管这一项，一般用默认的就好。**
* elementwise_affine：当为True时， γ 和 β 参数是可学习的；反之，是不可学习的。

输入输出的维度：

输入维度： (N,∗)

输出维度： (N,∗)

In [6]:
# NLP Example
embedding = torch.tensor([[[1,2,3,4]]], dtype=torch.float32)
batch, sentence_length, embedding_dim = embedding.shape  # 1,1,4
mean = embedding.mean(dim=-1, keepdim=True)  # 2.5
std = embedding.std(dim=-1, keepdim=True)    # 1.2910
layer_norm = nn.LayerNorm(embedding_dim)
# Activate module
norm_embedding = layer_norm(embedding)       # [[[-1.3416, -0.4472,  0.4472,  1.3416]]]
print("mean:", mean)
print("std:", std)
print("layer embdedding:", norm_embedding)

mean: tensor([[[2.5000]]])
std: tensor([[[1.2910]]])
layer embdedding: tensor([[[-1.3416, -0.4472,  0.4472,  1.3416]]],
       grad_fn=<NativeLayerNormBackward0>)


In [7]:
# Image Example
input = torch.tensor([[[[ 1,  2],
                        [3,  4]],
                       [[ 3,  4],
                        [2,  2]]]], dtype=torch.float32)
N, C, H, W = input.shape   # [N, C, H, W]: [1,2,2,2]
# Normalize over the last one dimensions (i.e feat dimension)
layer_norm = nn.LayerNorm(W)
output1 = layer_norm(input)
# Normalize over the last two dimensions(i.e  spatial dimensions)
layer_norm = nn.LayerNorm([H,W])
output2 = layer_norm(input)
# Normalize over the last two dimensions(i.e  channel and spatial dimensions)
layer_norm = nn.LayerNorm([C, H, W])
output3 = layer_norm(input)
print("output1",output1)
print("output2",output2)
print("output3",output3)

output1 tensor([[[[-1.0000,  1.0000],
          [-1.0000,  1.0000]],

         [[-1.0000,  1.0000],
          [ 0.0000,  0.0000]]]], grad_fn=<NativeLayerNormBackward0>)
output2 tensor([[[[-1.3416, -0.4472],
          [ 0.4472,  1.3416]],

         [[ 0.3015,  1.5075],
          [-0.9045, -0.9045]]]], grad_fn=<NativeLayerNormBackward0>)
output3 tensor([[[[-1.6378, -0.6299],
          [ 0.3780,  1.3859]],

         [[ 0.3780,  1.3859],
          [-0.6299, -0.6299]]]], grad_fn=<NativeLayerNormBackward0>)


### InstanceNorm2D
- **归一化维度**: 针对单个样本的每个通道，在空间维度 𝐻×𝑊 上计算均值和方差。
- **公式**:

\begin{align*}
\mu_I=\frac{1}{H W} \sum_{i=1}^H \sum_{j=1}^W x_{n, c, i, j} \\ \sigma_I^2=\frac{1}{H W} \sum_{i=1}^H \sum_{j=1}^W\left(x_{n, c, i, j}-\mu_I\right)^2 \\ \hat{x}_{n, c, i, j}=\frac{x_{n, c, i, j}-\mu_I}{\sqrt{\sigma_I^2+\epsilon}} \\ y_{n, c, i, j}=\gamma_c \hat{x}_{n, c, i, j}+\beta_c
\end{align*}

它是**对每个样本输入的后两维（即除了batchsize维和channel维）合并在一起**算均值和方差的。比如输入大小为 (N,C,H,W) ，则均值 E[x] 为 input.mean((−2,−1))

![](https://ones.ainewera.com/wiki/api/wiki/editor/JNwe8qUX/RDYCnwvo/resources/RTJv__Qy5fxfwFHSXhMKWPWJ6TePjt0VNKOCSnYHFbc.png?token=W.h2ZbJ4tknyp15AZ3wAqFjz9AVke9xNe8MgBZ16uEBjRjHAXghPTeHOCSS3jem-A)

代码：

```
torch.nn.InstanceNorm2d(num_features, eps=1e-05, momentum=0.1, affine=False, track_running_stats=False, device=None, dtype=None)
```

主要参数的含义：

* num_features：通道数C。
* eps：归一化时加到分母上，防止分母过小导致数值计算不稳定。**不用管这一项，一般用默认的就好。**
* momentum：在track_running_stats为True时，momentum是训练过程中对均值和方差进行动量更新的动量参数；在track_running_stats为False时，momentum不起作用。
* affine：当为True时， γ 和 β 参数是可学习的；反之，是不可学习的。
* track_running_stats：当为True时，在训练时会始终记录并更新（通过动量方法更新）全局的均值和方差，然后在**测试**时可以用这个均值和方差来归一化（ **为什么要这样做？你可以理解为这个均值和方差是所有训练样本的均值和方差，是全局的，对整个样本集的统计信息的描述更加准确一些** ）；当为False时，不记录更新全局的均值和方差，这样的话，**测试**时用那个batch的测试数据本身的样本和方差来归一化。

输入输出的维度：

输入维度：(N,C,H,W)

输出维度：(N,C,H,W)

In [8]:
# NLP Example
embedding = torch.tensor([[[1,2,3,4]],[[2,2,3,4]]], dtype=torch.float32)
batch, sentence_length, embedding_dim = embedding.shape  # 2,1,4
batch_norm = nn.BatchNorm1d(sentence_length)
ins_norm = nn.InstanceNorm1d(sentence_length)

batch_embedding = batch_norm(embedding)
ins_embedding = ins_norm(embedding)
print("batch embdedding:", batch_embedding)
print("ins embdedding:", ins_embedding)

# instance norm的结果等于前面batch=1时batch_norm的结果，说明instance只是针对单个样本(单batch计算均值和方差)

batch embdedding: tensor([[[-1.6378, -0.6299,  0.3780,  1.3859]],

        [[-0.6299, -0.6299,  0.3780,  1.3859]]],
       grad_fn=<NativeBatchNormBackward0>)
ins embdedding: tensor([[[-1.3416, -0.4472,  0.4472,  1.3416]],

        [[-0.9045, -0.9045,  0.3015,  1.5075]]])


### GroupNorm2D
- **归一化维度**: 针对单个样本的分组通道（将通道分为 𝐺 组）和空间维度 𝐻×𝑊 一起计算均值和方差。
- **公式**:

\begin{align}
\mu_G &= \frac{1}{\text{group\_size} \cdot H \cdot W} \sum_{c \in \text{group}} \sum_{i=1}^H \sum_{j=1}^W x_{n, c, i, j}, \\ \sigma_G^2 &= \frac{1}{\text{group\_size} \cdot H \cdot W} \sum_{c \in \text{group}} \sum_{i=1}^H \sum_{j=1}^W \left(x_{n, c, i, j} - \mu_G\right)^2, \\ \hat{x}_{n, c, i, j} &= \frac{x_{n, c, i, j} - \mu_G}{\sqrt{\sigma_G^2 + \epsilon}}, \\ y_{n, c, i, j} &= \gamma_c \hat{x}_{n, c, i, j} + \beta_c.
\end{align}


它也是**对单个样本的输入的通道分组后，对输入的每组通道以及后面的维度合并在一起**算均值和方差的。 **如果分为C组，即每组一个channel，则等价于InstanceNorm；如果只分为1组，即所有channel为一组，则等价于LayerNorm** 。

![](https://ones.ainewera.com/wiki/api/wiki/editor/JNwe8qUX/RDYCnwvo/resources/4qv9ivoTYtuKKQmv1_W1LFleZuAtYSqJbfkd5QQcY3w.png?token=W.h2ZbJ4tknyp15AZ3wAqFjz9AVke9xNe8MgBZ16uEBjRjHAXghPTeHOCSS3jem-A)

代码：

```
torch.nn.GroupNorm(num_groups, num_channels, eps=1e-05, affine=True, device=None, dtype=None)
```

主要参数的含义：

* num_groups：通道分为几组。
* num_channels：通道数C。
* eps：归一化时加到分母上，防止分母过小导致数值计算不稳定。**不用管这一项，一般用默认的就好。**
* affine：当为True时， γ 和 β 参数是可学习的；反之，是不可学习的。

输入输出的维度：

输入维度：(N,C,∗)

输出维度：(N,C,∗)

In [13]:
# NLP Example
embedding = torch.tensor([[[1,2,3,4], [2,2,3,4]]], dtype=torch.float32)
batch, sentence_length, embedding_dim = embedding.shape  # 1,2,4
g1_norm = nn.GroupNorm(1,sentence_length)
g2_norm = nn.GroupNorm(2,sentence_length)
# Activate module
g1_embedding = g1_norm(embedding)
g2_embedding = g2_norm(embedding)
print("Gorup=1 embdedding:", g1_embedding)
print("Gourp=2 embdedding:", g2_embedding)

Gorup=1 embdedding: tensor([[[-1.6378, -0.6299,  0.3780,  1.3859],
         [-0.6299, -0.6299,  0.3780,  1.3859]]],
       grad_fn=<NativeGroupNormBackward0>)
Gourp=2 embdedding: tensor([[[-1.3416, -0.4472,  0.4472,  1.3416],
         [-0.9045, -0.9045,  0.3015,  1.5075]]],
       grad_fn=<NativeGroupNormBackward0>)


### 四种归一化方法的核心区别
| 特性                | **BatchNorm**                              | **InstanceNorm**                     | **LayerNorm**                        | **GroupNorm**                    |
|---------------------|--------------------------------------------|--------------------------------------|--------------------------------------|----------------------------------|
| **归一化维度**      | 对 mini-batch 的 **每个通道** 跨样本归一化 | 对单个样本的 **每个通道** 单独归一化 | 对单个样本的 **所有通道** 一起归一化 | 对单个样本的 **分组通道** 归一化 |
| **依赖 batch size** | 强依赖 (batch size 大效果更好)             | 无依赖                               | 无依赖                               | 无依赖                           |
| **适用场景**        | 图像分类等任务，需较大 batch size           | 风格迁移、生成对抗网络等任务          | NLP、时间序列数据等                   | 小 batch size 或需稳定性任务     |
| **归一化目标**      | 跨样本统计特征一致性                       | 每个样本保持独立                     | 平滑单个样本特征                     | 跨组平滑特征分布                 |
| **可调参数**        | 可学习缩放和偏移                           | 可学习缩放和偏移                     | 可学习缩放和偏移                     | 可学习缩放和偏移                 |

### 适用场景对比总结

| 归一化方法       | 适用场景                   | 优点                                  | 局限性                                   |
|------------------|----------------------------|---------------------------------------|------------------------------------------|
| **BatchNorm**    | 大 batch size 的卷积任务   | 高效稳定，能利用 mini-batch 的统计特性 | 对 batch size 依赖较强，小 batch 表现不佳 |
| **InstanceNorm** | 风格迁移、生成任务          | 样本独立归一化，适合生成任务           | 忽略了样本间的统计特性                   |
| **LayerNorm**    | NLP、RNN、小 batch size 任务 | 对 batch size 不敏感                  | 对于卷积任务不一定高效                   |
| **GroupNorm**    | 小 batch size 的卷积任务   | 对 batch size 不敏感，兼顾效率和性能   | 分组数量需要调节                         |

### RMSNorm
- **归一化维度**：同layerNorm, RMSNorm是对LayerNorm的改进，RMSNorm认为re-centering invariance property是不必要的，只用保留re-scaling invariance property。
- **公式**

\begin{align*}
\text{RMS}(x) &= \sqrt{\frac{1}{d} \sum_{i=1}^d x_i^2}, \\
\hat{x} &= \frac{x}{\text{RMS}(x) + \epsilon}\gamma, \\
\end{align*}

代码

```
torch.nn.RMSNorm(normalized_shape, eps=1e-8, elementwise_affine=True)
```

参数解释:
* normalized_shape: 指定被归一化的维度。例如，对于 2D 数据，可以指定特征维度大小。

* eps: 防止分母为零的小常数，默认为 1×10−8。

* elementwise_affine: 是否启用仿射变换（可学习的 𝛾 参数）。默认为 True。



In [ ]:
# NLP Example
embedding = torch.tensor([[[1,2,3,4],[5,6,7,8]]], dtype=torch.float32)
batch, sentence_length, embedding_dim = embedding.shape  # 1,2,4
mean = embedding.mean(dim=-1, keepdim=True)
std = embedding.std(dim=-1, keepdim=True)
rms_norm = nn.RMSNorm(embedding_dim)
# Activate module
norm_embedding = rms_norm(embedding)
print("mean:", mean)
print("std:", std)
print("rms embdedding:", norm_embedding)

In [ ]:
import time

# 设置设备
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 定义输入张量
batch_size, seq_len, feature_dim = 128, 64, 512  # 假设常用的 Transformer 输入大小
x = torch.randn(batch_size, seq_len, feature_dim, device=device)

# 定义 LayerNorm 和 RMSNorm
layer_norm = nn.LayerNorm(feature_dim).to(device)
rms_norm = nn.RMSNorm(feature_dim).to(device)

# 运行速度测试函数
def measure_time(norm_layer, x, num_runs=100):
    start_time = time.time()
    for _ in range(num_runs):
        _ = norm_layer(x)  # 执行归一化
    end_time = time.time()
    return (end_time - start_time) / num_runs

# 比较运行时间
layer_norm_time = measure_time(layer_norm, x)
rms_norm_time = measure_time(rms_norm, x)

# 输出结果
print(f"LayerNorm 平均运行时间: {layer_norm_time * 1000:.3f} ms")
print(f"RMSNorm 平均运行时间: {rms_norm_time * 1000:.3f} ms")

### 参考文档

[知乎文章](https://zhuanlan.zhihu.com/p/470260895)

[BatchNorm](https://arxiv.org/pdf/1502.03167.pdf)

[LayerNorm](https://arxiv.org/pdf/1607.06450.pdf)

[InstanceNorm](https://arxiv.org/pdf/1607.08022.pdf)

[GroupNorm](https://arxiv.org/pdf/1803.08494.pdf)

[RMSNorm](https://arxiv.org/abs/1910.07467)